In [2]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("group-by-joins") \
    .getOrCreate()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/13 15:47:32 WARN Utils: Your hostname, Adrians-M2-Macbook.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.97 instead (on interface en0)
26/03/13 15:47:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/13 15:47:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/13 15:47:34 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/13 15:47:34 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [3]:
# Read Parquet Files
df_green = spark.read.parquet("../data/parquet_spark/green/*/*/*.parquet")
df_yellow = spark.read.parquet("../data/parquet_spark/yellow/*/*/*.parquet")

26/03/13 15:47:40 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: ../data/parquet_spark/green/*/*/*.parquet.
java.io.FileNotFoundException: File ../data/parquet_spark/green/*/*/*.parquet does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at or

In [4]:
# Create Temporary Views
df_green.createOrReplaceTempView("green")
df_yellow.createOrReplaceTempView("yellow")

In [17]:
df_green_revenue = spark.sql("""
    SELECT 
        -- Revenue grouping 
        date_trunc('hour', lpep_pickup_datetime) AS hour,
        PULocationID AS revenue_zone,

        COUNT(*) AS record_count,
        SUM(total_amount) AS total_amount
    FROM green
    WHERE lpep_pickup_datetime >= '2020-01-01'
    GROUP BY 1, 2
    ORDER BY 1, 2
""")

df_green_revenue.show(5)




+-------------------+------------+------------+-----------------+
|               hour|revenue_zone|record_count|     total_amount|
+-------------------+------------+------------+-----------------+
|2020-01-01 00:00:00|           7|          45|769.7299999999996|
|2020-01-01 00:00:00|          17|           9|           195.03|
|2020-01-01 00:00:00|          18|           1|              7.8|
|2020-01-01 00:00:00|          22|           1|             15.8|
|2020-01-01 00:00:00|          24|           3|             87.6|
+-------------------+------------+------------+-----------------+
only showing top 5 rows


In [6]:
df_green_revenue \
    .repartition(20) \
    .write.parquet("../data/report/revenue/green", mode="overwrite")

26/03/13 15:48:29 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/13 15:48:29 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/13 15:48:30 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/03/13 15:48:30 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/03/13 15:48:30 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/03/13 15:48:30 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/03/13 15:48:30 WARN MemoryManager: Total allocation exceeds 95.

In [21]:
df_yellow_revenue = spark.sql("""
    SELECT 
        -- Revenue grouping 
        date_trunc('hour', tpep_pickup_datetime) AS hour,
        PULocationID AS revenue_zone,

        COUNT(*) AS record_count,
        SUM(total_amount) AS total_amount
    FROM yellow
    WHERE tpep_pickup_datetime >= '2020-01-01'
    GROUP BY 1, 2
    ORDER BY 1, 2
""")

df_yellow_revenue.show(5)

+-------------------+------------+------------+-----------------+
|               hour|revenue_zone|record_count|     total_amount|
+-------------------+------------+------------+-----------------+
|2020-01-01 00:00:00|           3|           1|             25.0|
|2020-01-01 00:00:00|           4|          57|           1004.3|
|2020-01-01 00:00:00|           7|          38|455.1700000000002|
|2020-01-01 00:00:00|          10|           2|            42.41|
|2020-01-01 00:00:00|          12|           6|            107.0|
+-------------------+------------+------------+-----------------+
only showing top 5 rows


In [20]:
df_yellow_revenue \
    .repartition(20) \
    .write.parquet("../data/report/revenue/yellow", mode="overwrite")


26/03/13 16:01:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/13 16:01:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/13 16:01:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/03/13 16:01:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/03/13 16:01:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/03/13 16:01:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/03/13 16:01:33 WARN MemoryManager: Total allocation exceeds 95.

In [9]:
df_green_revenue = spark.read.parquet("../data/report/revenue/green")
df_yellow_revenue = spark.read.parquet("../data/report/revenue/yellow")

In [10]:
df_green_revenue_tmp = df_green_revenue \
    .withColumnRenamed("total_amount", "green_amount") \
    .withColumnRenamed("record_count", "green_number_of_records") \

df_yellow_revenue_tmp = df_yellow_revenue \
    .withColumnRenamed("total_amount", "yellow_amount") \
    .withColumnRenamed("record_count", "yellow_number_of_records") 

In [11]:
df_join = df_green_revenue_tmp \
    .join(df_yellow_revenue_tmp, on=["hour", "revenue_zone"], how="outer") \
    .orderBy("hour")

df_join.show(5)


+-------------------+------------+-----------------------+------------+------------------------+-------------+
|               hour|revenue_zone|green_number_of_records|green_amount|yellow_number_of_records|yellow_amount|
+-------------------+------------+-----------------------+------------+------------------------+-------------+
|2002-12-31 23:00:00|         264|                   NULL|        NULL|                       1|          0.0|
|2002-12-31 23:00:00|         193|                   NULL|        NULL|                       1|          0.0|
|2003-01-01 00:00:00|          68|                   NULL|        NULL|                       1|         22.3|
|2003-01-01 00:00:00|         193|                   NULL|        NULL|                       1|          0.0|
|2003-01-01 00:00:00|         237|                   NULL|        NULL|                       1|         12.8|
+-------------------+------------+-----------------------+------------+------------------------+-------------+
o

In [12]:
df_join.write.parquet("../data/report/revenue/total", mode="overwrite")

26/03/13 15:51:16 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/13 15:51:16 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/13 15:51:16 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/03/13 15:51:16 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/03/13 15:51:16 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/03/13 15:51:16 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/03/13 15:51:16 WARN MemoryManager: Total allocation exceeds 95.

In [13]:
df_join = spark.read.parquet("../data/report/revenue/total")

In [ ]:
df_join.sort("hour", ascending=True).show()
What happens wh

+-------------------+------------+-----------------------+------------+------------------------+-------------+
|               hour|revenue_zone|green_number_of_records|green_amount|yellow_number_of_records|yellow_amount|
+-------------------+------------+-----------------------+------------+------------------------+-------------+
|2002-12-31 23:00:00|         193|                   NULL|        NULL|                       1|          0.0|
|2002-12-31 23:00:00|         264|                   NULL|        NULL|                       1|          0.0|
|2003-01-01 00:00:00|         193|                   NULL|        NULL|                       1|          0.0|
|2003-01-01 00:00:00|          68|                   NULL|        NULL|                       1|         22.3|
|2003-01-01 00:00:00|         237|                   NULL|        NULL|                       1|         12.8|
|2003-01-01 01:00:00|         140|                   NULL|        NULL|                       1|         30.3|
|